# MuseTalk 1.5 GPU validation worker

Runs MuseTalk 1.5 on a Colab T4 using an approved singer image/video and bhajan audio. The notebook is deliberately fail-fast: if inference fails, it stops at the real failing step instead of falsely reaching the download step.


In [ ]:
# 1. Runtime / repository setup
!nvidia-smi
!pip -q install uv

import os, subprocess
from pathlib import Path

MT = Path("/content/MuseTalk")
VENV = Path("/content/musetalk310")

if not MT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/TMElyralab/MuseTalk.git", str(MT)], check=True)

if not (VENV / "bin/python").exists():
    subprocess.run(["uv", "venv", "--python", "3.10", str(VENV)], check=True)

PY = str(VENV / "bin/python")
subprocess.run([PY, "-V"], check=True)
print("MuseTalk:", MT)
print("Python:", PY)


In [ ]:
# 2. Python 3.10 dependencies
import subprocess

PY = "/content/musetalk310/bin/python"
MT = "/content/MuseTalk"

subprocess.run(["uv", "pip", "install", "--python", PY, "pip", "setuptools", "wheel"], check=True)
subprocess.run([
    "uv", "pip", "install", "--python", PY,
    "torch==2.0.1", "torchvision==0.15.2", "torchaudio==2.0.2",
    "--index-url", "https://download.pytorch.org/whl/cu118"
], check=True)
subprocess.run(["uv", "pip", "install", "--python", PY, "-r", MT + "/requirements.txt"], check=True)
subprocess.run(["uv", "pip", "install", "--python", PY, "openmim"], check=True)
subprocess.run(["uv", "pip", "install", "--python", PY, "--no-build-isolation", "chumpy==0.70"], check=True)

MIM = "/content/musetalk310/bin/mim"
for pkg in ["mmengine", "mmcv==2.0.1", "mmdet==3.1.0", "mmpose==1.1.0"]:
    subprocess.run([MIM, "install", pkg], check=True)

print("MuseTalk dependencies installed.")


In [ ]:
# 3. Patch MuseTalk inference.py safely for this Colab worker.
# The upstream script catches exceptions and continues, and uses shell ffmpeg calls.
# We replace those behaviors so a failed inference cannot be mistaken for success.

from pathlib import Path
import re

inf = Path("/content/MuseTalk/scripts/inference.py")
s = inf.read_text()

s = s.replace(
    "os.system(cmd_img2video)",
    "subprocess.run(cmd_img2video, shell=True, check=True)"
)
s = s.replace(
    "os.system(cmd_combine_audio)",
    "subprocess.run(cmd_combine_audio, shell=True, check=True)"
)

s = s.replace(
    "            shutil.rmtree(save_dir_full)",
    "            if 'save_dir_full' in locals() and os.path.exists(save_dir_full):\n"
    "                shutil.rmtree(save_dir_full)"
)

s = re.sub(
    r'except Exception as e:\n\s+print(\"Error occurred during processing:\", e)',
    "except Exception:\n            raise",
    s
)

inf.write_text(s)
print("Patched:", inf)


In [ ]:
# 4. Download core model files (resumable)
import subprocess
from pathlib import Path

MODELS = Path("/content/MuseTalk/models")
MODELS.mkdir(parents=True, exist_ok=True)

def download_file(url, target, min_bytes):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)

    if target.exists() and target.stat().st_size >= min_bytes:
        print("✓ already ready:", target, target.stat().st_size, "bytes")
        return

    print("Downloading:", url)
    subprocess.run([
        "curl", "-L", "--fail", "--retry", "8", "--retry-delay", "5",
        "-C", "-", "-o", str(target), url
    ], check=True)

    size = target.stat().st_size
    if size < min_bytes:
        raise RuntimeError(f"Incomplete download: {target} ({size} bytes)")

download_file(
    "https://huggingface.co/TMElyralab/MuseTalk/resolve/main/musetalkV15/unet.pth?download=true",
    MODELS / "musetalkV15/unet.pth", 3_000_000_000
)
download_file(
    "https://huggingface.co/TMElyralab/MuseTalk/resolve/main/musetalkV15/musetalk.json?download=true",
    MODELS / "musetalkV15/musetalk.json", 500
)
download_file(
    "https://huggingface.co/stabilityai/sd-vae-ft-mse/resolve/main/config.json?download=true",
    MODELS / "sd-vae/config.json", 500
)
download_file(
    "https://huggingface.co/stabilityai/sd-vae-ft-mse/resolve/main/diffusion_pytorch_model.bin?download=true",
    MODELS / "sd-vae/diffusion_pytorch_model.bin", 300_000_000
)
download_file(
    "https://huggingface.co/openai/whisper-tiny/resolve/main/config.json?download=true",
    MODELS / "whisper/config.json", 500
)
download_file(
    "https://huggingface.co/openai/whisper-tiny/resolve/main/preprocessor_config.json?download=true",
    MODELS / "whisper/preprocessor_config.json", 1_000
)
download_file(
    "https://huggingface.co/openai/whisper-tiny/resolve/main/pytorch_model.bin?download=true",
    MODELS / "whisper/pytorch_model.bin", 100_000_000
)

print("Core weights ready.")


In [ ]:
# 5. Download auxiliary model files (resumable)
import subprocess
from pathlib import Path

MODELS = Path("/content/MuseTalk/models")

def fetch(url, target, min_bytes):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)

    if target.exists() and target.stat().st_size >= min_bytes:
        print("✓ already ready:", target, target.stat().st_size, "bytes")
        return

    print("Downloading:", url)
    subprocess.run([
        "curl", "-L", "--fail", "--retry", "8", "--retry-delay", "5",
        "-C", "-", "-o", str(target), url
    ], check=True)

    size = target.stat().st_size
    if size < min_bytes:
        raise RuntimeError(f"Incomplete download: {target} ({size} bytes)")

fetch(
    "https://huggingface.co/ManyOtherFunctions/face-parse-bisent/resolve/main/79999_iter.pth?download=true",
    MODELS / "face-parse-bisent/79999_iter.pth", 40_000_000
)
fetch(
    "https://download.pytorch.org/models/resnet18-5c106cde.pth",
    MODELS / "face-parse-bisent/resnet18-5c106cde.pth", 40_000_000
)
fetch(
    "https://huggingface.co/yzd-v/DWPose/resolve/main/dw-ll_ucoco_384.pth?download=true",
    MODELS / "dwpose/dw-ll_ucoco_384.pth", 300_000_000
)

print("Auxiliary models ready.")


In [ ]:
# 6. Upload the approved singer image/video and the bhajan audio
from google.colab import files
from pathlib import Path

print("Upload the APPROVED singer image/video:")
uploaded_avatar = files.upload()
if not uploaded_avatar:
    raise RuntimeError("No avatar uploaded.")
avatar = next(iter(uploaded_avatar))

print("Upload the successful ACE-Step bhajan MP3/WAV:")
uploaded_audio = files.upload()
if not uploaded_audio:
    raise RuntimeError("No audio uploaded.")
audio = next(iter(uploaded_audio))

assert Path(avatar).suffix.lower() in {
    ".png", ".jpg", ".jpeg", ".webp", ".mp4", ".mov", ".webm"
}, f"Unsupported avatar: {avatar}"

assert Path(audio).suffix.lower() in {
    ".mp3", ".wav", ".m4a", ".flac", ".aac", ".ogg"
}, f"Unsupported audio: {audio}"

print("Avatar:", avatar)
print("Audio:", audio)


In [ ]:
# 7. Normalize inputs: 25-fps MP4 + 16-kHz mono WAV
import subprocess
from pathlib import Path

OUT = Path("/content/musetalk_output")
OUT.mkdir(parents=True, exist_ok=True)

avatar_video = OUT / "avatar_input.mp4"
audio_wav = OUT / "audio.wav"

subprocess.run([
    "ffmpeg", "-y", "-i", audio,
    "-ar", "16000", "-ac", "1", str(audio_wav)
], check=True)

probe = subprocess.run([
    "ffprobe", "-v", "error",
    "-show_entries", "format=duration",
    "-of", "default=noprint_wrappers=1:nokey=1",
    str(audio_wav)
], capture_output=True, text=True, check=True)

duration = float(probe.stdout.strip())

suffix = Path(avatar).suffix.lower()
if suffix in {".png", ".jpg", ".jpeg", ".webp"}:
    subprocess.run([
        "ffmpeg", "-y",
        "-loop", "1", "-i", avatar,
        "-t", str(duration),
        "-r", "25",
        "-vf", "scale=512:-2,format=yuv420p",
        "-an", str(avatar_video)
    ], check=True)
else:
    subprocess.run([
        "ffmpeg", "-y",
        "-i", avatar,
        "-t", str(duration),
        "-r", "25",
        "-vf", "scale=512:-2,format=yuv420p",
        "-an", str(avatar_video)
    ], check=True)

assert avatar_video.exists() and avatar_video.stat().st_size > 10_000
assert audio_wav.exists() and audio_wav.stat().st_size > 10_000

print("Avatar video:", avatar_video)
print("Audio WAV:", audio_wav)
print("Duration:", round(duration, 2), "seconds")


In [ ]:
# 8. Create the MuseTalk task configuration
from pathlib import Path

MT = Path("/content/MuseTalk")
OUT = Path("/content/musetalk_output")
cfg = MT / "configs/inference/test.yaml"

cfg.write_text(
    "bhajan_test:\n"
    f"  video_path: '{OUT / 'avatar_input.mp4'}'\n"
    f"  audio_path: '{OUT / 'audio.wav'}'\n"
    "  result_name: 'bhajan_lipsync.mp4'\n"
)

print(cfg.read_text())


In [ ]:
# 9. Run MuseTalk 1.5
# IMPORTANT: this cell is the actual inference step.
# It will stop on the real traceback if MuseTalk fails.

import os, subprocess
from pathlib import Path

os.chdir("/content/MuseTalk")

env = os.environ.copy()
env["MPLBACKEND"] = "Agg"
env["PYTHONPATH"] = "/content/MuseTalk:" + env.get("PYTHONPATH", "")

result_dir = Path("/content/musetalk_output/result")
result_dir.mkdir(parents=True, exist_ok=True)

expected = result_dir / "v15" / "bhajan_lipsync.mp4"
if expected.exists():
    expected.unlink()

cmd = [
    "/content/musetalk310/bin/python", "-m", "scripts.inference",
    "--inference_config", "configs/inference/test.yaml",
    "--result_dir", str(result_dir),
    "--unet_model_path", "models/musetalkV15/unet.pth",
    "--unet_config", "models/musetalkV15/musetalk.json",
    "--whisper_dir", "models/whisper",
    "--version", "v15",
    "--fps", "25",
    "--batch_size", "4",
    "--use_float16",
    "--parsing_mode", "jaw",
]

print("Starting MuseTalk 1.5 on T4...")
subprocess.run(cmd, env=env, check=True)

if not expected.exists() or expected.stat().st_size < 100_000:
    print("\nResult directory contents:")
    for p in sorted(result_dir.rglob("*")):
        if p.is_file():
            print(p, p.stat().st_size, "bytes")
    raise RuntimeError(
        "MuseTalk completed but the expected MP4 was not created: "
        + str(expected)
    )

print("SUCCESS:", expected)
print("Size:", round(expected.stat().st_size / 1024 / 1024, 2), "MB")


In [ ]:
# 10. Download final MP4
from pathlib import Path
from google.colab import files

candidate = Path("/content/musetalk_output/result/v15/bhajan_lipsync.mp4")

if not candidate.exists():
    raise RuntimeError(f"Expected MP4 not found: {candidate}")

assert candidate.stat().st_size > 100_000, "MP4 is unexpectedly small"

print("SUCCESS:", candidate)
print("Size:", round(candidate.stat().st_size / 1024 / 1024, 2), "MB")

files.download(str(candidate))
